In [6]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="ollama:gemma4",
    # the checkpointer works great for tracking shorter conversations, but over time, this can grow too big and overflow the agent's context
    # when this happens, the agent cannot efficiently go through all the context, which in turn slows down the agent's response time, worsens the performance (reduces accuracy), and increases costs!
    # there are 2 ways you can to solve this problem: summarising the conversation or trimming the conversation.
    checkpointer=InMemorySaver(),
    # middleware are control loops, which allow us to control and customise our agent's execution
    # an example of middleware could be a classifier that check's the agent's output and decides the next step, e.g. a legal agent could have custom middleware () that decides if something is legal or not legal
    # custom middleware docs: https://docs.langchain.com/oss/python/langchain/middleware/custom
    # another example of middleware could be a refund agent - in this case, a human in the loop middleware could be used to check if the agent's output is correct before proceeding with the refund
    # human-in-the-loop middleware docs: https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop
    middleware=[
        # this will send the summary as HumanMessage():
        # [HumanMessage(content='Here is a summary of the conversation to date...). After this message, another HumanMessage() will be sent to ask the final question "If you were Lunapolis' new president...". The agent will respond (as AIMessage()).
        SummarizationMiddleware(
            model="ollama:gemma4:e2b", # the model that will be used to summarise the conversation
            trigger=("tokens", 100), # the number of tokens that once reached will trigger summarisation of the conversation
            keep=("messages", 1) # the number of messages to keep after summarising - all other messages will be deleted
        )
    ],
)

In [8]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user was engaged in a series of hypothetical questions about the fictional location "Lunapolis," seeking information regarding its capital, weather, population statistics (specifically cheese miners), and political/labor conditions (union strike).\n\n## SUMMARY\n\nKey facts established about Lunapolis:\n*   **Capital:** Lunapolis.\n*   **Weather:** Skies are clear, with a high of 120C and a low of -100C.\n*   **Population:** 100,000 cheese miners live in Lunapolis.\n*   **Union Status:** The cheese miners\' union is predicted to strike because they are unhappy with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNone', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='74187462-7e2d-4eec-b642-d7b87bf3df83'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", addit

In [9]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user was engaged in a series of hypothetical questions about the fictional location "Lunapolis," seeking information regarding its capital, weather, population statistics (specifically cheese miners), and political/labor conditions (union strike).

## SUMMARY

Key facts established about Lunapolis:
*   **Capital:** Lunapolis.
*   **Weather:** Skies are clear, with a high of 120C and a low of -100C.
*   **Population:** 100,000 cheese miners live in Lunapolis.
*   **Union Status:** The cheese miners' union is predicted to strike because they are unhappy with the new president.

## ARTIFACTS

None

## NEXT STEPS

None


## Trim/delete messages

In [10]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

# we can create custom middleware using decorators, see here: https://docs.langchain.com/oss/python/langchain/middleware/custom#node-style-hooks
# custom middleware can be configured to run before or after and the agent, or before or after a model. See above link for details.
# removing tool messages can be useful if they don't provide any useful information and are just cluttering up the context window.
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [11]:
agent = create_agent(
    model="ollama:gemma4",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages], # we can then reference the custom middleware, just as with the pre-built middleware earlier
)

In [12]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?") # this question will confirm that the ToolMessage stating the temperature has been removed
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='cc06df10-d2b9-4c5f-b542-9b742a7377b8'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='28bb61c9-b382-4fe4-9079-5a881e8422fd', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='f92a5817-480c-46ba-91b5-62f73497acf5'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='73c24d63-a7dd-46c6-bc9d-9a40e80b7387', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='258c3a22-1a6a-4016-9504-6f1c25c92acb'),
              AIMessage(content='I am an AI and cannot physically check the temperature of your

In [13]:
print(response["messages"][-1].content)

I am an AI and cannot physically check the temperature of your device.

**If you are concerned about the temperature, please follow these guidelines:**

* **Is it unusually hot?** If the device is extremely hot to the touch, **immediately unplug it** and do not attempt to turn it on again until it has cooled down completely. Excessive heat can indicate a serious internal problem.
* **Is it cold?** If it's normal room temperature, that's usually fine.

---

**However, since your initial problem was that the device won't turn on, let's focus on troubleshooting that. Since you confirmed it is plugged in and turned on, and assuming it is not overheating, here are the next steps:**

**1. Check the Power Source:**
* **Try a different outlet:** Plug the device into a completely different wall outlet to rule out an issue with the original socket or circuit breaker.
* **Check the power cord:** Ensure the cord is securely plugged into both the wall/adapter and the device. Inspect the cord for an